In [6]:
import torch
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torch.nn as nn
from tqdm import tqdm, trange
from pytorch_model import ModelTrainer

In [7]:
# Transformatie om afbeeldingen naar PyTorch Tensors te converteren
transform = transforms.Compose([transforms.ToTensor()])

# Download en laad de MNIST train- en testdataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Maak DataLoaders aan voor batches en shuffling

GPU_index = 0
device = torch.device(GPU_index if torch.cuda.is_available() else 'cpu')


train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [0.8, 0.2])

# DataLoaders aanmaken
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [8]:
print(len(train_loader))

750


In [59]:
res_net = models.resnet18(weights="IMAGENET1K_V1")

res_net

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [60]:
inputs, labels = next(iter(train_loader))
print(len(labels.unique()))

num_features = res_net.fc.in_features
res_net.fc = nn.Linear(num_features, 10, bias=True)

res_net.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

res_net = res_net.to(device)

10


In [61]:
#freeze the parameters
#for par in res_net.parameters():
    #par.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(res_net.parameters(), lr=0.001)

In [62]:
def training_loop(model, optimizer, criterion, loss_logg, loader, device):
    for inputs, labels in tqdm(loader, leave=False, desc='Training'):
        optimizer.zero_grad()
        preds = model(inputs.to(device))
        loss = criterion(preds, labels.to(device))

        loss_logg.append(loss.item())
        loss.backward()

        optimizer.step()
    return optimizer, loss_logg, model

def evaluate_loop(model,criterion, loss_logg, loader, device):
    total_acc = 0
    total_batch = 0
    for inputs, labels in tqdm(loader, leave=False, desc='Testing'):
        preds = model(inputs.to(device))
        loss = criterion(preds, labels.to(device))
        loss_logg.append(loss.item())

        total_acc += (torch.argmax(preds, dim=1) == labels.to(device)).sum().item()
        total_batch = labels.shape[0]
    return (total_acc/total_batch)*100

In [63]:
loss_logg = []
val_acc = 0
train_acc = 0
for epoch in trange(2, leave=False, desc='Epochs'):
    optimizer, loss_logg, res_net = training_loop(res_net, optimizer, criterion, loss_logg, train_loader, device)

    val_acc = evaluate_loop(res_net,criterion, loss_logg, val_loader, device)
    train_acc = evaluate_loop(res_net,criterion, loss_logg, train_loader, device)

    test_acc = evaluate_loop(res_net,criterion, loss_logg, test_loader, device)
    print(val_acc)
    print(train_acc)
    print(test_acc)

Training: 100%|██████████| 750/750 [01:09<00:00, 10.80it/s]
                                                           
Testing:  99%|█████████▉| 187/188 [00:06<00:00, 29.50it/s]
                                                          
Testing: 100%|██████████| 750/750 [00:25<00:00, 30.37it/s]
                                                          
Epochs:  50%|█████     | 1/2 [01:46<01:46, 106.70s/it]    

36246.875
72882.8125
60900.0



Training: 100%|██████████| 750/750 [01:12<00:00, 10.11it/s]
                                                           
Testing:  98%|█████████▊| 185/188 [00:06<00:00, 27.73it/s]
                                                          
Testing: 100%|██████████| 750/750 [00:26<00:00, 29.34it/s]
                                                          
Testing:  99%|█████████▉| 156/157 [00:05<00:00, 29.51it/s]
                                                          

36703.125
73806.25
61425.0
